# Day 20 — Model Serving & Inference Optimization
### Cloud fallback (Google Colab / Kaggle)

**Chỉ dùng notebook này nếu laptop của bạn không chạy được lab** — dưới 8 GB RAM, hoặc
setup thất bại vì lý do bạn không sửa được. Lab được thiết kế cho máy của bạn; toàn bộ
ý nghĩa của phần đo lường là đo **máy bạn**.

**Điểm không bị ảnh hưởng.** Rubric chấm độ rõ ràng của đo lường và lập luận, không chấm
tốc độ tuyệt đối. Nhưng bạn **phải khai báo** ở REFLECTION §1 rằng mình dùng cloud
fallback và vì sao — notebook tự ghi `runtime_environment` vào `hardware.json` cho bạn.

Notebook này chạy đúng các script như bản laptop và sinh ra **đúng các tên file**, nên
`make verify` và rubric không cần đường riêng cho cloud. Cuối cùng bạn tải một file zip
về và commit vào repo của mình.

> **Kaggle:** bật **Internet ON** trong Settings ở sidebar, nếu không bước tải model sẽ fail.

> **Runtime menu → Change runtime type:** CPU là mặc định và đủ dùng. GPU T4 nhanh hơn
> nhưng cần cell build CUDA (~8 phút) ở mục 4b.


## 1. Cấu hình


In [ ]:
# Repo chứa code lab. Mặc định là repo gốc (public) nên notebook chạy được ngay.
# Muốn clone từ fork của bạn thì đổi dòng này — không bắt buộc: artifact được sinh
# ra trong VM rồi bạn tải zip về, nên nguồn clone không ảnh hưởng tới bài submit.
REPO_URL = "https://github.com/VinUni-AI20k/Day20-Track2-ModelServing.git"

# Model: 'qwen35-0.8b' (~0.9 GB) hoặc 'gemma4-e2b' (~5.2 GB).
# Trên VM 2 vCPU của Colab/Kaggle, model nhỏ chạy nhanh hơn nhiều và tải nhanh hơn.
LAB_MODEL = "qwen35-0.8b"

# Colab đã chiếm port 8080 (service riêng của Colab), nên lab dùng port khác.
# Trên laptop bạn không cần dòng này — mặc định của lab là 8080.
LAB_SERVER_PORT = "8090"

# 'cpu'  = binary CPU prebuilt. Luôn chạy được. Chậm hơn nhưng đủ cho cả lab.
# 'cuda' = tự compile llama.cpp với CUDA (~8 phút). Chỉ khi bạn chọn GPU runtime.
#          llama.cpp KHÔNG publish prebuilt CUDA cho Linux, nên phải build.
RUNTIME = "cpu"

# Load test ngắn hơn để session free-tier không hết giờ.
LOAD_DURATION = "1m"


## 2. Nhận diện môi trường + helper `sh()`

`sh()` chạy lệnh và in ra output. Dùng Python thuần thay vì `!magic` để notebook hoạt
động giống nhau trên Colab và Kaggle.


In [ ]:
import os, sys, subprocess, pathlib, time

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
ENV = 'colab' if IN_COLAB else 'kaggle' if IN_KAGGLE else 'unknown'
WORK = pathlib.Path('/content' if IN_COLAB else '/kaggle/working' if IN_KAGGLE else '.')

# Đây là thứ được ghi vào hardware.json và thoả rubric item 1.
os.environ['LAB_RUNTIME_ENV'] = ENV
os.environ['LAB_MODEL'] = LAB_MODEL
os.environ['LAB_SERVER_PORT'] = LAB_SERVER_PORT


def sh(*args, check=True, quiet=False):
    """Chạy một lệnh và IN OUTPUT RA CELL. Trả về exit code.

    Output của process con được đọc qua pipe rồi print bằng Python. Bắt buộc phải
    làm vậy: notebook không capture những gì process con ghi thẳng ra file
    descriptor, nên nếu không pipe thì cell sẽ trống và bạn không có gì để
    screenshot.

    check=False: lệnh fail (hoặc không tồn tại) không làm hỏng cell. Dùng cho các
    lệnh chỉ để xem thông tin — vd nvidia-smi không có trên CPU runtime.
    """
    cmd = [str(a) for a in args]
    if not quiet:
        print('$', ' '.join(cmd), flush=True)
    try:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
    except (FileNotFoundError, OSError) as exc:
        if check:
            raise SystemExit(f'không chạy được {cmd[0]}: {exc}')
        print(f'  (bỏ qua: {cmd[0]} không có trên máy này)')
        return 127
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise SystemExit(f'lệnh thất bại (exit {rc}): {" ".join(cmd)}')
    return rc


def py(*args, **kw):
    """Chạy một script Python của lab bằng đúng interpreter đang dùng."""
    return sh(sys.executable, '-u', *args, **kw)


print('môi trường :', ENV)
print('workdir    :', WORK)
sh('nproc', check=False)
sh('free', '-g', check=False)
sh('nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader', check=False)


## 3. Clone lab + cài dependency

Bốn package Python thuần. Không compiler, không `llama-cpp-python`.


In [ ]:
LAB = WORK / 'Day20-Track2-ModelServing'
if not LAB.exists():
    sh('git', 'clone', '--depth', '1', REPO_URL, LAB)
os.chdir(LAB)
print('cwd:', os.getcwd())

py('-m', 'pip', 'install', '-q', '-r', 'requirements.txt')
print('deps đã cài')


## 4. Probe hardware + tải llama.cpp runtime


In [ ]:
py('labs/00-setup/detect-hardware.py')

if RUNTIME == 'cpu':
    # Buộc dùng asset CPU. Auto-picker sẽ chọn bản Vulkan khi thấy GPU NVIDIA, nhưng
    # image Colab/Kaggle thường không có Vulkan driver nên bản đó không bind được device.
    sys.path.insert(0, 'lib')
    import labkit
    asset = f'llama-{labkit.LLAMA_CPP_BUILD}-bin-ubuntu-x64.tar.gz'
    py('labs/00-setup/fetch-runtime.py', '--asset', asset)
else:
    print('RUNTIME=cuda -> bỏ qua bước tải prebuilt; chạy cell build CUDA bên dưới.')


### 4b. (Optional) Build với CUDA — chỉ khi dùng GPU runtime, ~8 phút

Bỏ qua cell này nếu `RUNTIME = 'cpu'`. Chạy nó cũng là làm được bonus **B1** (nhớ chạy
`bonus/compare-builds.py` sau đó — B1 yêu cầu **so sánh**, không phải chỉ build), và bạn
sẽ có sẵn cả CUDA build lẫn prebuilt để làm challenge **C6**.


In [ ]:
if RUNTIME == 'cuda':
    # Gọi cmake trực tiếp: các target trong Makefile cần .venv, còn notebook này cài
    # vào system Python.
    sh('apt-get', '-qq', 'install', '-y', 'cmake', 'build-essential')
    sys.path.insert(0, 'lib')
    import labkit
    BUILD = labkit.LLAMA_CPP_BUILD
    if not pathlib.Path('bonus/llama.cpp').exists():
        sh('git', 'clone', '--depth', '1', '--branch', BUILD,
           'https://github.com/ggml-org/llama.cpp', 'bonus/llama.cpp')
    sh('cmake', '-B', 'bonus/llama.cpp/build', '-S', 'bonus/llama.cpp',
       '-DGGML_CUDA=ON', '-DGGML_NATIVE=ON', '-DCMAKE_BUILD_TYPE=Release')
    sh('cmake', '--build', 'bonus/llama.cpp/build', '-j', '--config', 'Release')
    sh('ls', '-la', 'bonus/llama.cpp/build/bin')
    print('CUDA build xong — mọi script của lab tự tìm thấy nó.')
else:
    print('bỏ qua (RUNTIME != cuda)')


## 5. Tải model

Notebook này mặc định `LAB_MODEL = 'qwen35-0.8b'` —
**[unsloth/Qwen3.5-0.8B-GGUF](https://huggingface.co/unsloth/Qwen3.5-0.8B-GGUF)**, ~0.9 GB, Apache-2.0, **không gated**.
Đổi sang `'gemma4-e2b'` trong cell 1 nếu bạn muốn model lớn hơn (~5.2 GB).

Hai quantization của model đã chọn: primary + compare (rubric item 3 cần cả hai hàng).

Nếu bước này fail, script in ra đúng lệnh `curl` để tải tay.


In [ ]:
py('labs/00-setup/download-model.py')
print(pathlib.Path('models/active.json').read_text())


## 6. Track 01 — Measure

**Screenshot output của cả hai cell** (ảnh số 2, và ảnh tune optional).

Trên 2 vCPU việc này chậm — vài phút cho mỗi quantization. Bản thân điều đó cũng là một
quan sát đáng viết vào reflection.


In [ ]:
py('labs/01-measure/benchmark.py')


In [ ]:
# VM cloud ít core nên grid thread rất ngắn (thường chỉ [1, 2]) và spread nhỏ.
# Vẫn là số thật của máy đang chạy, và phần giải thích mới là chỗ được chấm:
# nếu peak nằm ở logical core chứ không phải physical core, hãy nói rõ và lý giải.
py('labs/01-measure/tune.py')


## 7. Track 02 — Serve

Server chạy background trong notebook, đúng như nó sẽ chạy ở terminal thứ hai trên laptop.

> **Số đo thật trên Colab CPU runtime (1 physical / 2 logical core):** một request
> 48 token mất khoảng **6–7 s**, tức decode ~8–10 tok/s. Vì chỉ có 1 core, thêm slot
> **không** tăng throughput — trần thực tế là **~0.15 request/s**. Trong 1 phút bạn
> chỉ hoàn thành khoảng **7–10 request** mỗi lần load test.
>
> Điều đó **không** làm hỏng bài: rubric cần hai mức concurrency và một saturation
> reading, và trên máy chậm hiệu ứng saturation còn *rõ hơn*. Bằng chứng mạnh nhất
> không phải percentile mà là `requests_deferred` ở cell metrics: trên VM này nó lên
> tới **46 request đang xếp hàng** sau 4 slot. Đó chính là queue time mà §8 nói tới.
>
> Percentile sẽ mỏng (ít mẫu) và `load-report` sẽ tự cảnh báo. Muốn số chắc hơn thì
> đổi `LOAD_DURATION = '3m'` ở cell 1 — đổi lấy thời gian chạy lâu hơn.


In [ ]:
import urllib.request

# Server log goes to a file, not DEVNULL: if llama-server dies at startup we need to
# show the student WHY instead of silently polling for 300s.
srv_log_path = '/tmp/llama-server.log'
srv_log = open(srv_log_path, 'w')
srv = subprocess.Popen([sys.executable, 'labs/02-serve/serve.py'],
                       stdout=srv_log, stderr=subprocess.STDOUT)

def _server_log(n=25):
    try:
        srv_log.flush()
    except Exception:
        pass
    try:
        return ''.join(open(srv_log_path).readlines()[-n:])
    except OSError:
        return '(khong doc duoc log)'

health = f'http://127.0.0.1:{LAB_SERVER_PORT}/health'
ok = False
for i in range(300):
    if srv.poll() is not None:          # process da chet -> dung cho het 300s
        raise RuntimeError(
            f'serve.py thoat som (exit {srv.returncode}). Log cua server:\n' + _server_log())
    try:
        with urllib.request.urlopen(health, timeout=2) as r:
            if r.status == 200:
                print(f'server healthy sau {i}s tren port {LAB_SERVER_PORT}')
                ok = True
                break
    except Exception:
        pass
    time.sleep(1)

if not ok:
    srv.terminate()                     # dung de lai process treo
    raise RuntimeError(
        f'server khong len sau 300s tren port {LAB_SERVER_PORT}. Log cua server:\n'
        + _server_log())

In [ ]:
# Rubric item 6 + 7 — screenshot output này.
py('labs/02-serve/smoke-test.py')


In [ ]:
# Rubric item 8 — screenshot bảng summary.
py('-m', 'locust', '-f', 'labs/02-serve/load-test.py', '--headless',
   '-u', '10', '-r', '5', '-t', LOAD_DURATION, '--host', f'http://localhost:{LAB_SERVER_PORT}',
   '--csv', 'benchmarks/locust-10', '--only-summary')


In [ ]:
# Chạy load 50 user VÀ sample /metrics cùng lúc (item 9 cần sự chồng thời gian này).
rec = subprocess.Popen([sys.executable, 'labs/02-serve/record-metrics.py',
                        '--duration', '60', '--label', 'u50'])
py('-m', 'locust', '-f', 'labs/02-serve/load-test.py', '--headless',
   '-u', '50', '-r', '25', '-t', LOAD_DURATION, '--host', f'http://localhost:{LAB_SERVER_PORT}',
   '--csv', 'benchmarks/locust-50', '--only-summary')
rec.wait()


In [ ]:
# Rubric item 10.
py('labs/02-serve/load-report.py')


## 8. Track 03 — Integrate


In [ ]:
py('labs/03-integrate/pipeline.py')


## 9. Verify, rồi mang artifact về máy

`verify` sẽ báo thiếu REFLECTION và screenshots — bạn điền hai thứ đó trên máy mình.

> Trên Colab, `verify` cũng báo `... exists but is NOT committed` cho `hardware.json`
> và các file `locust-*_stats.csv`. **Điều này là bình thường ở đây**: clone trong VM
> không phải repo của bạn. Sau khi bạn giải nén zip vào clone local và `git add`,
> các dòng đó sẽ hết.

Mọi thứ trong `benchmarks/`, cộng `hardware.json` và `models/active.json`, là phần bạn commit.


In [ ]:
try:
    srv.terminate()          # tắt server nếu session này có bật
except NameError:
    pass

rc = py('scripts/verify.py', check=False)
print(f'(verify exit {rc} — báo thiếu REFLECTION/screenshots là bình thường ở đây)')
print()
sh('ls', '-la', 'benchmarks')


In [ ]:
# Zip CHỈ phần evidence — không bao giờ zip ~5 GB weights.
import zipfile

ZIP = WORK / 'day20-artifacts.zip'
ZIP.unlink(missing_ok=True)
wanted = ['hardware.json', 'models/active.json']
wanted += [str(p.relative_to(LAB)) for p in sorted((LAB / 'benchmarks').rglob('*'))
           if p.is_file() and p.name != '.gitkeep']

with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for rel in wanted:
        src = LAB / rel
        if src.exists():
            z.write(src, rel)
            print('  +', rel)
        else:
            print('  ! thiếu (bước đó đã chạy chưa?):', rel)

print(f'\nxong: {ZIP}  ({ZIP.stat().st_size / 1024:.0f} KB)')
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))
else:
    print('Kaggle: tìm file trong panel Output bên phải.')


## 10. Hoàn tất trên máy của bạn

1. Giải nén vào clone local: `benchmarks/`, `hardware.json`, `models/active.json`.
2. Thay **mọi** section `"required -- replace this line"` trong `benchmarks/*.md` bằng
   nhận xét của bạn. `make verify` sẽ fail nếu còn sót.
3. Điền `submission/REFLECTION.md`. **Ở §1 nói rõ bạn dùng cloud fallback và vì sao** —
   `hardware.json` đã ghi `runtime_environment` sẵn.
4. Add 5 screenshots (chụp từ output các cell của notebook này).
5. `make verify` → exit 0, push lên repo **public**, paste URL vào LMS.

### Một điều đáng viết vào §5

VM cloud không phải laptop của bạn: khác số core, khác memory bandwidth, có hypervisor ở
giữa, và có neighbour tranh tài nguyên trên cùng host. Số của bạn **đúng cho VM này** và
không so được với laptop của bạn cùng lớp — điều đó vốn đã đúng với cả bản laptop. Nói rõ
trong §5 rằng kết quả tuning của bạn mô tả cái VM bạn được cấp. Nhận ra được giới hạn đó
chính là loại lập luận rubric thưởng điểm.

Nếu session bị disconnect giữa đường: chạy lại từ mục 3. Clone và tải model là hai bước
chậm duy nhất, và cả hai đều bỏ qua phần đã có trên đĩa.
